# Data Analyst (итерация 1)

# Data Analyst Report: Fake Job Postings — EDA

**Бизнес-задача:** бинарная классификация мошеннических вакансий (`fraudulent`) на HR-площадке. Цель — снизить ручную модерацию и защитить соискателей от скам-постингов. Приоритетная метрика — **F1 / recall класса 1** при контроле precision.

**Что покажет EDA:**
1. Общая структура очищенного датасета (shape, dtypes, группы колонок).
2. Распределение таргета и степень дисбаланса.
3. Корреляции числовых / бинарных признаков с `fraudulent` и топ-сигналы.
4. Mean target rate по топ-категориям категориальных признаков (`*_freq` и one-hot).
5. Сигнал из текстовых полей: длина текста и доля пустых описаний в зависимости от класса.

Все графики собираются в список `FIGS` (plotly), все ключевые числа печатаются в stdout для дальнейшего анализа.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

FIGS = []

DF = pd.read_csv('/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv')

print('SHAPE:', DF.shape)
print('\nDTYPES VALUE COUNTS:')
print(DF.dtypes.value_counts())
print('\nHEAD:')
print(DF.head(3))
print('\nNaN total:', DF.isna().sum().sum())

SHAPE: (17880, 75)

DTYPES VALUE COUNTS:
int64      66
str         5
float64     4
Name: count, dtype: int64

HEAD:
   job_id                                      title                                    company_profile                                        description  \
0  179.79                           Marketing Intern  We're Food52, and we've created a groundbreaki...  Food52, a fast-growing, James Beard Award-winn...   
1  179.79  Customer Service - Cloud Video Production  90 Seconds, the worlds Cloud Video Production ...  Organised - Focused - Vibrant - Awesome!Do you...   
2  179.79    Commissioning Machinery Assistant (CMA)  Valor Services provides Workforce Solutions th...  Our client, located in Houston, is actively se...   

                                        requirements                                           benefits  telecommuting  has_company_logo  has_questions  fraudulent  employment_type_Contract  \
0  Experience with content management systems a m...      

## 1. Обзор датасета

Разобьём колонки на группы: **target**, **бинарные флаги**, **числовые (в т.ч. `*_freq`)**, **one-hot категориальные**, **сырой текст**. Выведем сводки по каждой группе и `describe()` для числовых.

In [ ]:
TARGET = 'fraudulent'

text_like_candidates = ['title', 'description', 'requirements', 'benefits', 'company_profile']
# dtype у строковых колонок может быть object ИЛИ pandas StringDtype ('string'/'str') —
# надёжнее проверять, что колонка не числовая.
text_cols = [c for c in text_like_candidates
             if c in DF.columns and not pd.api.types.is_numeric_dtype(DF[c])]

num_cols_all = DF.select_dtypes(include=[np.number]).columns.tolist()
num_cols_all = [c for c in num_cols_all if c != TARGET]

# Разделим числовые на бинарные флаги, freq-encoded и прочие
binary_flags = [c for c in num_cols_all if DF[c].dropna().isin([0, 1]).all() and DF[c].nunique() <= 2]
freq_cols = [c for c in num_cols_all if c.endswith('_freq')]
# one-hot = бинарные с префиксом исходной категории (обычно содержат '_')
onehot_cols = [c for c in binary_flags if c not in ['telecommuting', 'has_company_logo', 'has_questions']]
real_binary_flags = [c for c in binary_flags if c in ['telecommuting', 'has_company_logo', 'has_questions']]
other_num = [c for c in num_cols_all if c not in binary_flags and c not in freq_cols]

print(f'TARGET: {TARGET}')
print(f'TEXT columns ({len(text_cols)}): {text_cols}')
print(f'BINARY FLAGS ({len(real_binary_flags)}): {real_binary_flags}')
print(f'FREQ-ENCODED ({len(freq_cols)}): {freq_cols}')
print(f'OTHER NUMERIC ({len(other_num)}): {other_num[:10]}{" ..." if len(other_num)>10 else ""}')
print(f'ONE-HOT columns: {len(onehot_cols)} (показываем 10): {onehot_cols[:10]}')

print('\nDESCRIBE (real binary flags + freq + other numeric):')
cols_to_describe = real_binary_flags + freq_cols + other_num
print(DF[cols_to_describe].describe().T[['count','mean','std','min','50%','max']])


TARGET: fraudulent
TEXT columns (5): ['title', 'description', 'requirements', 'benefits', 'company_profile']
BINARY FLAGS (3): ['telecommuting', 'has_company_logo', 'has_questions']
FREQ-ENCODED (3): ['location_freq', 'department_freq', 'industry_freq']
OTHER NUMERIC (1): ['job_id']
ONE-HOT columns: 62 (показываем 10): ['employment_type_Contract', 'employment_type_Full-time', 'employment_type_Other', 'employment_type_Part-time', 'employment_type_Temporary', 'required_experience_Associate', 'required_experience_Director', 'required_experience_Entry level', 'required_experience_Executive', 'required_experience_Internship']

DESCRIBE (real binary flags + freq + other numeric):
                    count         mean          std         min          50%           max
telecommuting     17880.0     0.042897     0.202631    0.000000     0.000000      1.000000
has_company_logo  17880.0     0.795302     0.403492    0.000000     1.000000      1.000000
has_questions     17880.0     0.491723     0

## 2. Распределение целевой переменной

Таргет `fraudulent` — бинарный. Ожидаем сильный дисбаланс (~5% позитивов). Это напрямую влияет на выбор метрик (F1/recall/PR-AUC, не accuracy) и стратегию обучения.

In [ ]:
vc = DF[TARGET].value_counts().sort_index()
total = len(DF)
pos = int(vc.get(1, 0))
neg = int(vc.get(0, 0))
pos_rate = pos / total
imbalance_ratio = neg / max(pos, 1)

print('TARGET value_counts:')
print(vc)
print(f'\nTotal rows: {total}')
print(f'Positives (fraudulent=1): {pos} ({pos_rate:.4%})')
print(f'Negatives (fraudulent=0): {neg} ({neg/total:.4%})')
print(f'Imbalance ratio (neg/pos): {imbalance_ratio:.2f} : 1')

# Plot 1: bar
fig1 = px.bar(x=['not fraud (0)', 'fraud (1)'], y=[neg, pos],
              title=f'Target distribution — fraudulent (positives: {pos_rate:.2%})',
              labels={'x': 'class', 'y': 'count'}, text=[neg, pos], color=['not fraud','fraud'])
fig1.update_traces(textposition='outside')
FIGS.append(fig1)

# Plot 2: pie
fig2 = px.pie(values=[neg, pos], names=['not fraud (0)', 'fraud (1)'],
              title='Target share — fraudulent', hole=0.4)
FIGS.append(fig2)
print(f'\nFIGS so far: {len(FIGS)}')

TARGET value_counts:
fraudulent
0    17014
1      866
Name: count, dtype: int64

Total rows: 17880
Positives (fraudulent=1): 866 (4.8434%)
Negatives (fraudulent=0): 17014 (95.1566%)
Imbalance ratio (neg/pos): 19.65 : 1

FIGS so far: 2


## 3. Числовые признаки vs target

Посчитаем корреляцию Пирсона всех числовых колонок (включая бинарные флаги и `*_freq`) с таргетом, отсортируем по модулю и покажем топ-кандидатов в сигнал.

In [ ]:
num_for_corr = real_binary_flags + freq_cols + other_num
corr_with_target = DF[num_for_corr + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET)
corr_sorted = corr_with_target.reindex(corr_with_target.abs().sort_values(ascending=False).index)

print('TOP-15 numeric features by |corr| with target:')
print(corr_sorted.head(15).round(4))
print('\nBOTTOM-5 (weakest):')
print(corr_sorted.tail(5).round(4))

# Heatmap корреляций топ-12 + target
top_for_heat = corr_sorted.head(12).index.tolist() + [TARGET]
corr_mat = DF[top_for_heat].corr(numeric_only=True)
fig3 = px.imshow(corr_mat, text_auto='.2f', color_continuous_scale='RdBu_r',
                 zmin=-1, zmax=1, title='Correlation heatmap — top-12 numeric features + target',
                 aspect='auto')
FIGS.append(fig3)

# Bar |corr| top-15
top15 = corr_sorted.head(15)
fig4 = px.bar(x=top15.values, y=top15.index, orientation='h',
              title='Top-15 |correlation| with fraudulent',
              labels={'x': 'corr with target', 'y': 'feature'},
              color=top15.values, color_continuous_scale='RdBu_r')
fig4.update_layout(yaxis={'categoryorder': 'total ascending'})
FIGS.append(fig4)
print(f'\nFIGS so far: {len(FIGS)}')

TOP-15 numeric features by |corr| with target:
has_company_logo   -0.2620
has_questions      -0.0916
job_id              0.0795
location_freq      -0.0460
telecommuting       0.0345
department_freq    -0.0161
industry_freq      -0.0031
Name: fraudulent, dtype: float64

BOTTOM-5 (weakest):
job_id             0.0795
location_freq     -0.0460
telecommuting      0.0345
department_freq   -0.0161
industry_freq     -0.0031
Name: fraudulent, dtype: float64

FIGS so far: 2


In [ ]:
# Распределение топ-2 числовых (не бинарных) признаков по классам target
continuous_like = [c for c in corr_sorted.index if c in freq_cols + other_num]
top2_cont = continuous_like[:2] if len(continuous_like) >= 2 else continuous_like
print(f'TOP-2 continuous-like features for per-class distribution: {top2_cont}')

for col in top2_cont:
    by_cls = DF.groupby(TARGET)[col].agg(['mean', 'median', 'std', 'count'])
    print(f'\n[{col}] stats by target class:')
    print(by_cls.round(4))

if len(top2_cont) >= 1:
    col = top2_cont[0]
    fig5 = px.histogram(DF, x=col, color=DF[TARGET].astype(str), nbins=60, barmode='overlay',
                        opacity=0.6, histnorm='probability density',
                        title=f'Distribution of {col} by target class (density)')
    FIGS.append(fig5)

if len(top2_cont) >= 2:
    col = top2_cont[1]
    fig5b = px.histogram(DF, x=col, color=DF[TARGET].astype(str), nbins=60, barmode='overlay',
                         opacity=0.6, histnorm='probability density',
                         title=f'Distribution of {col} by target class (density)')
    FIGS.append(fig5b)

# Бинарные флаги vs target — тоже важная таблица
if real_binary_flags:
    print('\nMean target rate by binary flags:')
    for f in real_binary_flags:
        grp = DF.groupby(f)[TARGET].agg(['mean', 'count'])
        print(f'\n[{f}]')
        print(grp.round(4))

print(f'\nFIGS so far: {len(FIGS)}')

TOP-2 continuous-like features for per-class distribution: ['job_id', 'location_freq']

[job_id] stats by target class:
                  mean  median        std  count
fraudulent                                      
0            8847.9896  8926.5  5090.7857  17014
1           10758.0195  9434.5  6069.0316    866

[location_freq] stats by target class:
              mean  median     std  count
fraudulent                               
0           0.0078  0.0019  0.0114  17014
1           0.0053  0.0011  0.0083    866

Mean target rate by binary flags:

[telecommuting]
                 mean  count
telecommuting               
0              0.0469  17113
1              0.0834    767

[has_company_logo]
                    mean  count
has_company_logo               
0                 0.1593   3660
1                 0.0199  14220

[has_questions]
                 mean  count
has_questions               
0              0.0678   9088
1              0.0284   8792

FIGS so far: 2


## 4. Категориальные признаки

После очистки высокочастотные категории представлены как `*_freq` (frequency encoding), а низкочастотные — one-hot. Посмотрим топ-группы по признакам с наибольшей связью с таргетом и посчитаем **mean target rate** внутри них.

In [ ]:
# Берём топ-3 FREQ-колонки по |corr| с target как кандидатов-категориальных
freq_by_corr = [c for c in corr_sorted.index if c in freq_cols][:3]
print(f'TOP-3 freq-encoded features by |corr|: {freq_by_corr}')

# Для каждой: биним по квантилям значений частоты и смотрим mean target
for col in freq_by_corr:
    try:
        bins = pd.qcut(DF[col], q=10, duplicates='drop')
    except Exception as e:
        print(f'[{col}] qcut failed: {e}')
        continue
    grp = DF.groupby(bins, observed=True)[TARGET].agg(['mean', 'count']).reset_index()
    grp[col] = grp[col].astype(str)
    print(f'\n[{col}] target rate by frequency-decile:')
    print(grp.round(4).to_string(index=False))
    fig = px.bar(grp, x=col, y='mean', title=f'Mean fraud rate by {col} (freq-decile)',
                 hover_data=['count'], labels={'mean': 'fraud rate'})
    fig.update_xaxes(tickangle=-30)
    FIGS.append(fig)

print(f'\nFIGS so far: {len(FIGS)}')

TOP-3 freq-encoded features by |corr|: ['location_freq', 'department_freq', 'industry_freq']

[location_freq] target rate by frequency-decile:
         location_freq   mean  count
(-0.0009441, 0.000112] 0.0829   2776
  (0.000112, 0.000224] 0.0461   1192
  (0.000224, 0.000447] 0.0437   1557
  (0.000447, 0.000895] 0.0465   1636
   (0.000895, 0.00185] 0.0376   1809
    (0.00185, 0.00442] 0.0452   1902
    (0.00442, 0.00733] 0.0368   1822
      (0.00733, 0.014] 0.0240   1665
       (0.014, 0.0264] 0.0718   2145
      (0.0264, 0.0402] 0.0160   1376

[department_freq] target rate by frequency-decile:
      department_freq   mean  count
(-0.0009441, 0.00028] 0.0581   1877
   (0.00028, 0.00268] 0.0768   1719
    (0.00268, 0.0272] 0.0375   2186
      (0.0272, 0.646] 0.0449  12098

[industry_freq] target rate by frequency-decile:
        industry_freq   mean  count
(-0.0009441, 0.00352] 0.0352   1901
    (0.00352, 0.0071] 0.0511   1878
     (0.0071, 0.0191] 0.1355   1742
     (0.0191, 0.0436] 0.

In [ ]:
# Один график — mean target по топ-10 one-hot категориям с наибольшим |corr|
if onehot_cols:
    oh_corr = corr_sorted.reindex([c for c in corr_sorted.index if c in onehot_cols]).dropna()
    top_oh = oh_corr.head(10)
    print('TOP-10 one-hot categories by |corr| with target (with mean target rate when flag=1):')
    rows = []
    for c in top_oh.index:
        mask = DF[c] == 1
        cnt = int(mask.sum())
        mt = DF.loc[mask, TARGET].mean() if cnt > 0 else np.nan
        base = DF[TARGET].mean()
        lift = (mt / base) if (cnt > 0 and base > 0) else np.nan
        rows.append({'feature': c, 'corr': top_oh[c], 'count_eq1': cnt,
                     'mean_target_when_1': mt, 'lift_vs_base': lift})
    oh_df = pd.DataFrame(rows)
    if not oh_df.empty and 'mean_target_when_1' in oh_df.columns:
        oh_df = oh_df.sort_values('mean_target_when_1', ascending=False)
    print(oh_df.round(4).to_string(index=False))

    if not oh_df.empty and oh_df['mean_target_when_1'].notna().any():
        fig = px.bar(oh_df, x='mean_target_when_1', y='feature', orientation='h',
                     color='lift_vs_base', color_continuous_scale='Reds',
                     hover_data=['count_eq1', 'corr'],
                     title='Top-10 one-hot categories: mean fraud rate when flag=1')
        fig.update_layout(yaxis={'categoryorder': 'total ascending'})
        FIGS.append(fig)
    else:
        print('No plottable one-hot rows (all counts=0 or empty).')
else:
    print('No one-hot columns detected.')

print(f'\nFIGS so far: {len(FIGS)}')


TOP-10 one-hot categories by |corr| with target (with mean target rate when flag=1):
Empty DataFrame
Columns: []
Index: []
No plottable one-hot rows (all counts=0 or empty).

FIGS so far: 0


## 5. Текстовые признаки

Для сырого текста считаем длину в словах, сравниваем распределения по классам и смотрим долю «пустых» (импутированных как `""`) текстов — это частый сигнал для фрод-постингов (короткие / отсутствующие описания компании).

In [ ]:
def word_count(s):
    if not isinstance(s, str):
        return 0
    return len(s.split())

wc_summary = {}
empty_rates = []
for col in text_cols:
    wc = DF[col].astype('object').fillna('').map(word_count)
    mean_by_cls = pd.DataFrame({TARGET: DF[TARGET], 'wc': wc}).groupby(TARGET)['wc'].agg(['mean', 'median', 'std'])
    wc_summary[col] = mean_by_cls
    # empty rate
    is_empty = DF[col].astype('object').fillna('').str.strip().eq('')
    er_by_cls = pd.DataFrame({TARGET: DF[TARGET], 'empty': is_empty.astype(int)}).groupby(TARGET)['empty'].mean()
    empty_rates.append({'col': col,
                        'empty_rate_overall': float(is_empty.mean()),
                        'empty_rate_notfraud': float(er_by_cls.get(0, np.nan)),
                        'empty_rate_fraud': float(er_by_cls.get(1, np.nan))})
    print(f'\n[{col}] word_count stats by target:')
    print(mean_by_cls.round(2))

if empty_rates:
    empty_df = pd.DataFrame(empty_rates)
    print('\nEMPTY (=="") rate by text column and class:')
    print(empty_df.round(4).to_string(index=False))
else:
    empty_df = pd.DataFrame(columns=['col', 'empty_rate_overall', 'empty_rate_notfraud', 'empty_rate_fraud'])
    print('\nNo text columns detected — skipping empty-rate table.')

# Plot: длина description (самый длинный текст) по классам
primary_text = 'description' if 'description' in text_cols else (text_cols[0] if text_cols else None)
if primary_text is not None:
    wc_primary = DF[primary_text].astype('object').fillna('').map(word_count)
    tmp = pd.DataFrame({'wc': wc_primary, TARGET: DF[TARGET].astype(str)})
    # clip длинный хвост для читаемости (не меняем DF)
    clip_val = int(np.quantile(tmp['wc'], 0.99))
    tmp['wc_clip'] = tmp['wc'].clip(upper=clip_val)
    fig9 = px.histogram(tmp, x='wc_clip', color=TARGET, nbins=60, barmode='overlay',
                        opacity=0.6, histnorm='probability density',
                        title=f'Word count of "{primary_text}" by target class (clipped at p99={clip_val})')
    FIGS.append(fig9)
else:
    print('\nNo primary text column — skipping word-count histogram.')

# Plot: empty rate по колонкам и классам
if not empty_df.empty:
    long = empty_df.melt(id_vars=['col'], value_vars=['empty_rate_notfraud', 'empty_rate_fraud'],
                         var_name='class', value_name='empty_rate')
    fig10 = px.bar(long, x='col', y='empty_rate', color='class', barmode='group',
                   title='Empty-text rate per column, split by target class',
                   labels={'col': 'text column', 'empty_rate': 'share of empty texts'})
    FIGS.append(fig10)
else:
    print('No empty-rate data to plot.')

print(f'\nFIGS so far: {len(FIGS)}')



[title] word_count stats by target:
            mean  median   std
fraudulent                    
0           3.75     3.0  2.06
1           4.02     3.0  2.31

[description] word_count stats by target:
              mean  median     std
fraudulent                        
0           171.04   147.0  122.56
1           158.75   113.5  136.63

[requirements] word_count stats by target:
             mean  median    std
fraudulent                      
0           79.03    63.0  81.93
1           58.41    34.0  73.36

[benefits] word_count stats by target:
             mean  median    std
fraudulent                      
0           30.02     6.0  49.56
1           29.45     5.0  53.44

[company_profile] word_count stats by target:
             mean  median    std
fraudulent                      
0           95.65    86.0  85.69
1           31.71     0.0  52.47

EMPTY (=="") rate by text column and class:
            col  empty_rate_overall  empty_rate_notfraud  empty_rate_fraud
         

## 6. Итоги EDA

**Структура отчёта (для supervisor / analyze-ноды):**

1. **Shape & dtypes** — подтверждение, что очистка прошла: нет NaN, все табличные фичи числовые, текст сохранён как строка.
2. **Target** — количественно оценён дисбаланс (positives/negatives, imbalance ratio). Это обосновывает выбор F1/recall, а не accuracy, и стратегию балансировки *внутри* CV (не на уровне EDA).
3. **Numeric / binary flags** — топ-15 фичей по |corr| с `fraudulent`. Видны и `*_freq` (industry/location/…), и исходные флаги (`has_company_logo`, `telecommuting`, `has_questions`).
4. **Per-class distributions** — overlapping-гистограммы для топ-2 непрерывных признаков показывают, есть ли реальный сдвиг распределения у класса «fraud».
5. **Categorical signal** — mean target rate по децилям frequency-encoded колонок + топ-10 one-hot категорий с наибольшим lift относительно базового fraud rate.
6. **Text signal** — (a) длина `description` / `company_profile` у мошеннических vs нормальных вакансий; (b) доля пустых текстов — ключевой proxy для низкокачественных постингов.

**Что дальше (зона DS, НЕ EDA):**
- TF-IDF / embeddings на `description`, `requirements`, `company_profile`, `benefits`, `title`.
- Модели с учётом class imbalance (`class_weight`, `scale_pos_weight`, focal loss).
- Out-of-fold target encoding для высококардинальных полей, если `*_freq` окажется недостаточным.
- Валидация по F1/recall @ фиксированный precision на стратифицированных фолдах.

Все числа, напечатанные в stdout выше, будут подхвачены автоматическим анализом для формирования бизнес-инсайтов.